# 호르무즈 해협 크롤링 & 분석                                                                                          

### 라이브러리 불러오기

In [8]:
import pandas as pd # 데이터 조작 및 분석을 모듈
import time # 코드 실행 속도 조절을 위한 모듈
import re   # 정규표현식을 사용하여 텍스트에서 특수문자를 제거하거나 특정 패턴을 찾을 때 쓰는 모듈
import datetime # 날짜와 시간을 다루는 파이썬 기본 라이브러리

from tqdm import tqdm  # 진행상황 바 출력을 위한 라이브러리
from selenium import webdriver # 브라우저 자동화를 위한 모듈
from bs4 import BeautifulSoup as BS # HTML 내용 파싱을 위한 모듈
from selenium.webdriver.common.by import By # 다양한 방법으로 엘리먼트를 찾기 위한 모듈
from selenium.webdriver.support.ui import WebDriverWait # 브라우저가 요소를 찾을 때까지 대기(Wait)해주는 모듈
from selenium.webdriver.support import expected_conditions as EC # '어떤 상태'가 될 때까지 기다릴지 조건을 정해주는(EC) 기능 임포트

### 스크롤 다운 함수 정의

In [9]:
# 네이버 검색 결과의 무한 스크롤 특성으로 인해 처음 페이지에 접속하면 네이버는 서버 부하를 줄이기 위해 약 30개 정도의 글만 먼저 보여줌. 
# 따라서 사용자가 화면을 아래로 스크롤해야 추가로 글을 더 불러올 수 있음. -> 스코롤 다운 함수를 정의해 블로그 확보 갯수 조절 가능!

# 스크롤 다운 함수 정의
def doScrollDown(whileSeconds): 
    start = datetime.datetime.now() # 함수가 실행된 '현재 시점'의 시/분/초 가져오기
    end = start + datetime.timedelta(seconds=whileSeconds) # 입력받은 초 만큼의 시간 간격 생성
    while True:
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);') # 페이지 맨 아래로 스크롤 다운
        time.sleep(1.2) # 다음 목록이 로딩될 시간(약 1.2초)을 줌
        if datetime.datetime.now() > end: # 만약 현재 시간이 아까 계산해둔 종료 시간(end)보다 커지면 반복 break
            break 

### 브라우저(크롬) 실행 + 창 최대화

In [10]:
driver = webdriver.Chrome()
driver.maximize_window()

## '호르무즈 해협' 크롤링

### 호르무즈 해협을 둘러싼 이슈가 많은 기간(26.02.01~26.04.10)의 '호르무즈 해협' 네이버 블로그 크롤링

In [ ]:
# 26.02.01 ~ (미국, 이란 전쟁 발발일 : 26.02.28 포함) 26.04.10(현재)로 기간 설정된 '호르무즈 해협'키워드에 대한 네이버 블로그 url
naver_blog_url = 'https://search.naver.com/search.naver?ssc=tab.blog.all&query=%ED%98%B8%EB%A5%B4%EB%AC%B4%EC%A6%88%20%ED%95%B4%ED%98%91&sm=tab_opt&nso=so%3Ar%2Cp%3Afrom20260201to20260410'
driver.get(naver_blog_url)
time.sleep(2) # 검색 결과 로딩 대기

In [ ]:
# 목록 확장을 위해 스크롤
doScrollDown(20)

In [6]:
# [네이버 블로그 제목과 URL 추출 시작]

# 제목과 URL을 저장할 리스트 초기화 (나중에 카페와 구분하기 위해 blog_ 접두사 사용)
blog_title_list = []
blog_url_list = []

# 네이버가 최근 블로그 검색 결과 레이아웃을 업데이트하면서 클래스명이 무작위 문자열이 섞인 형태로 바뀐것을 F12키를 활용해 인지했습니다.
# 이런 클래스명은 네이버가 코드를 업데이트할 때마다 수시로 바뀌기 때문에, '클래스 이름' 대신 해당 클래스의 '구조'나 '속성'을 이용해 크롤링하는 것이 좋다고 느꼈습니다.
# <a> 태그 중에서 'data-heatmap-target' 속성값이 '.nblg'인 것만 골라내기
found_blog_elements = driver.find_elements(By.CSS_SELECTOR, 'a[data-heatmap-target=".nblg"]')

# 찾아낸 여러 개의 요소(found_blog_elements)를 하나씩 꺼내어 반복문 돌리기
for element in found_blog_elements:
    title_text = element.text.strip()
    url_link = element.get_attribute('href')
    
    if title_text and url_link: # 제목과 주소가 모두 정상적으로 존재할 때만 리스트에 추가 (빈 값 방지)
        blog_title_list.append(title_text) # 제목 리스트에 추가
        blog_url_list.append(url_link)     # 주소 리스트에 추가

# 중복 제거 (데이터 정제)
# 대량 수집 시 중복된 글을 피하기 위해 판다스 데이터프레임을 활용합니다.
df_blog_temp = pd.DataFrame({'title': blog_title_list, 'url': blog_url_list})

# subset=['url'] = 주소(url) 컬럼을 기준으로 중복된 블로그인지 검사해"라는 뜻
# keep='first' = 만약 똑같은 주소가 여러 개 발견되면, 가장 처음에 나온 것 하나만 남기고 나머지는 버려"라는 뜻
df_blog_temp = df_blog_temp.drop_duplicates(subset=['url'], keep='first')

# 중복 제거된 데이터를 다시 우리가 사용할 리스트로 변환
blog_title_list = df_blog_temp['title'].tolist()
blog_url_list = df_blog_temp['url'].tolist()

print(f"✅ 총 {len(blog_url_list)}개의 중복 없는 네이버 블로그 주소를 확보했습니다.")

✅ 총 540개의 중복 없는 네이버 블로그 주소를 확보했습니다.


In [13]:
from tqdm import tqdm # 진행상황 파악을 위한 tqdm 라이브러리 추가

# --- [네이버 블로그 상세 데이터 수집: 본문, 좋아요, 댓글 수, 댓글 내용] ---
# 1. 수집한 데이터를 담을 리스트 초기화
blog_new_doc = []      # 각 블로그의 전체 본문 텍스트
blog_like_cnt = []     # 공감(좋아요) 숫자
blog_comment_cnt = []  # 댓글 개수 숫자
blog_comment_list = [] # 댓글 텍스트 내용 (분석용)

# 2. 앞서 수집한 blog_url_list(블로그 주소 목록)를 순회하며 개별 페이지 접속
# 기존의 range(len(...))을 tqdm으로 감싸 시각적인 진행 바를 표시합니다.
for i in tqdm(range(len(blog_url_list)), desc="네이버 블로그 수집 진행 중"): 
    # [Tip] 새 탭을 열어 접속하는 방식은 메인 목록 탭을 유지할 수 있어 안정적입니다.
    driver.execute_script(f"window.open('{blog_url_list[i]}')")
    driver.switch_to.window(driver.window_handles[1]) # 새로 열린 탭(인덱스 1번)으로 제어권 이동
    
    # 페이지 로딩을 기다리기 위한 명시적 대기(Explicit Wait) 설정
    wait = WebDriverWait(driver, 3) 
    
    try:
        # [핵심] 네이버 블로그는 본문이 'mainFrame'이라는 iframe 안에 들어있습니다.
        # 이 프레임으로 전환하지 않으면 본문 안의 요소를 절대 찾을 수 없습니다.
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, 'mainFrame')))
        
        # 현재 프레임의 HTML 소스를 가져와 BeautifulSoup으로 파싱
        html = driver.page_source
        soup = BS(html, 'html.parser')
        
        # --- (1) 본문 텍스트 추출 ---
        try:
            # 네이버 블로그의 본문 영역 클래스는 보통 '.se-main-container'입니다. (구 버전은 #postViewArea)
            content = soup.select_one('.se-main-container, #postViewArea').get_text(strip=True)
        except:
            # 본문을 찾지 못하는 예외 상황(글 삭제, 비공개 등) 처리
            content = "본문 없음"
        blog_new_doc.append(content)

        # --- (2) 좋아요(공감) 수 추출 ---
        try:
            # [Tip] 공감 버튼은 화면 하단에 있어, 스크롤을 끝까지 내려야 데이터가 활성화되는 경우가 많습니다.
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") 
            
            # 좋아요 숫자가 담긴 요소를 기다림
            like_el = WebDriverWait(driver, 2).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "span.u_likeit_text._count.num"))
            )
            # 'textContent' 속성은 숨겨져 있거나 화면에 보이지 않는 텍스트도 강제로 가져옵니다.
            like_val = like_el.get_attribute('textContent').strip()
            
            # 숫자가 아니거나 비어있으면 0으로 초기화 (데이터 정제)
            if not like_val or not like_val.isdigit(): 
                like_val = "0"
        except:
            like_val = "0"
        blog_like_cnt.append(like_val)

        # --- (3) 댓글 개수 추출 ---
        try:
            # 댓글 버튼에 표시된 숫자를 추출
            comment_num = driver.find_element(By.ID, "commentCount").get_attribute('textContent').strip()
            if not comment_num: 
                comment_num = "0"
        except:
            comment_num = "0"
        blog_comment_cnt.append(comment_num)

        # --- (4) 댓글 실제 내용 추출 ---
        try:
            # 댓글 버튼을 찾아 클릭해야 댓글 창이 로드됩니다.
            # 일반 click()보다 자바스크립트를 이용한 클릭이 가려진 요소 클릭에 더 강합니다.
            comment_btn = driver.find_element(By.ID, "commentCount") 
            driver.execute_script("arguments[0].click();", comment_btn) 
            time.sleep(1) # 댓글창이 서버로부터 데이터를 받아오는 시간 대기
            
            # 댓글 텍스트가 담긴 모든 요소(u_cbox_contents)를 수집
            comments = driver.find_elements(By.CLASS_NAME, "u_cbox_contents") 
            # 여러 개의 댓글을 줄바꿈(\n)으로 구분하여 하나의 문자열로 합침
            blog_comment_final = "\n".join([c.text for c in comments]) if comments else "댓글 없음" 
        except:
            # 댓글 기능이 비활성화되었거나 접근 불가능한 경우
            blog_comment_final = "댓글 막힘" 
        blog_comment_list.append(blog_comment_final)

    except Exception as e:
        # 특정 게시글 수집 중 예상치 못한 오류 발생 시 처리
        # tqdm 환경에서는 print 대신 tqdm.write를 사용하는 것이 진행 바 깨짐을 방지합니다.
        tqdm.write(f"⚠️ {i+1}번 글 수집 실패: {e}")
        blog_new_doc.append("에러 발생")
        blog_like_cnt.append("0")
        blog_comment_cnt.append("0")
        blog_comment_list.append("에러 발생")

    # [중요] 사용이 끝난 상세 탭을 닫고, 다시 목록 페이지가 있는 0번 탭으로 제어권 복구
    driver.close()
    driver.switch_to.window(driver.window_handles[0])

# 모든 수집이 끝나면 브라우저 종료 및 최종 메시지
driver.quit()
print("✅ 수집 완료!")

네이버 블로그 수집 진행 중: 100%|██████████| 540/540 [25:35<00:00,  2.84s/it]


✅ 수집 완료!


In [14]:
# 1. 수집된 리스트들을 하나의 데이터프레임(DataFrame)으로 결합
df_blog = pd.DataFrame({
    'title': blog_title_list,        # 글 제목
    'url': blog_url_list,            # 글 주소
    'doc': blog_new_doc,             # 본문 내용
    'like': blog_like_cnt,           # 좋아요(공감) 수
    'comment_cnt': blog_comment_cnt,  # 댓글 수
    'comment_list': blog_comment_list, # 댓글 내용
    'ch': 'naver',                   # 채널 구분 1 (네이버)
    'ch2': 'blog'                    # 채널 구분 2 (블로그)
})

# 2. 데이터 유효성 확인
# 수집 과정에서 본문이 비었거나 에러가 난 항목이 있는지 상위 5개를 출력해 확인합니다.
print("📊 [검토] 네이버 블로그 데이터 생성 완료 (상위 5행):")
print(df_blog.head())

# 3. 파일 저장 (파일명: 26.02.01~26.04.10_naver_blog_data.csv)
file_name = '26.02.01~26.04.10_naver_blog_data.csv'

# encoding='utf-8-sig'는 엑셀에서 한글이 깨지지 않게 방지해주는 설정입니다.
df_blog.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"✅ 파일 저장 완료: {file_name}")
print(f"총 {len(df_blog)}개의 블로그 데이터가 안전하게 기록되었습니다.")

📊 [검토] 네이버 블로그 데이터 생성 완료 (상위 5행):
                                 title  \
0  미국 이란 전쟁 30일 경과, 호르무즈 해협 지도로 본 현 상황   
1    호르무즈 해협 통행료가 얼마길래? 미국 이란 공동 징수 추진   
2   뉴욕 증시 트럼프 발언 속 혼조세, 호르무즈 해협 통행료 이슈   
3   트럼프 “호르무즈 해협 이름 바꾼다!?”… 이거 진짜 가능한가   
4       호르무즈 해협 봉쇄 이유와 영향, 유가 상승·증시 전망   

                                               url  \
0      https://blog.naver.com/3f-hoon/224233386857   
1   https://blog.naver.com/totoro3123/224246051445   
2    https://blog.naver.com/archimall/224239139022   
3  https://blog.naver.com/kimsong8106/224233276505   
4   https://blog.naver.com/dldnsgud91/224201238593   

                                                 doc like comment_cnt  \
0  ​미국과 이란의 전쟁이 지난 2월 28일에 발발해 한 달이 지났습니다.​빠르게 끝날...   16           4   
1  안녕하세요.​네이버 경제인플루언서 주식너부리입니다.​4월 8일(현지시각) 도널드 트...   41           4   
2  최근 중동 지역의 긴장감이 고조되면서 투자자들의 심리가 얼음판 위를 걷는 듯한 모습...   89           4   
3  ​​최근 외신 보도를 보면 도널드 트럼프 대통령이 중동의 핵심 요충지인 호르무즈 해...   25           2   
4  ​최근 뉴스에서 “호르무즈 해협 봉쇄

### 호르무즈 해협을 둘러싼 이슈가 많은 기간(26.02.01~26.04.10) '호르무즈 해협' 네이버 카페 크롤링

In [15]:
# 브라우저 실행 & 최대화
driver = webdriver.Chrome()
driver.maximize_window()

In [16]:
# 26.02.01 ~ (미국, 이란 전쟁 발발일 : 26.02.28 포함) 26.04.10(현재)로 기간 설정된 '호르무즈 해협'키워드에 대한 네이버 카페 url
# 네이버 카페 검색 옵션에서 '일반글' 선택으로 불필요한 데이터인 '거래글'수집 사전 차단.
naver_cafe_url = "https://search.naver.com/search.naver?cafe_where=articleg&nso=so%3Ar%2Cp%3Afrom20260201to20260410&nso_open=1&prdtype=0&query=%ED%98%B8%EB%A5%B4%EB%AC%B4%EC%A6%88+%ED%95%B4%ED%98%91&sm=mtb_opt&ssc=tab.cafe.all&st=rel&stnm=rel&opt_tab=0&nso=so%3Ar%2Cp%3Afrom20260201to20260410&date_from=20260201&date_to=20260410"
driver.get(naver_cafe_url)
time.sleep(2) # 페이지 로딩 대기

In [19]:
# 목록 확장을 위해 스크롤
doScrollDown(20)

In [20]:
# --- [네이버 카페 제목과 URL 추출 시작] ---

# 제목과 URL을 저장할 리스트 초기화 (나중에 블로그와 구분하기 위해 cafe_ 접두사 사용)
cafe_title_list = []
cafe_url_list = []

# BeautifulSoup으로 현재 페이지의 HTML 소스를 가져옵니다. (속도 최적화를 위해 BS 활용)
html = driver.page_source
soup = BS(html, 'html.parser')

# 네이버 카페 검색 결과의 게시글 요소들을 선택합니다.
# 카페 역시 클래스명이 바뀔 수 있으므로, 공통적인 구조인 '.view_wrap' 내부의 '.title_link'를 활용합니다.
found_cafe_elements = soup.select('.view_wrap .title_link')

# 찾아낸 여러 개의 요소들을 하나씩 꺼내어 반복문을 돌립니다.
for element in found_cafe_elements:
    title_text = element.get_text(strip=True)
    url_link = element['href']
    
    if title_text and url_link: # 제목과 주소가 모두 정상적으로 존재할 때만 리스트에 추가 (빈 값 방지)
        cafe_title_list.append(title_text) # 제목 리스트에 추가
        cafe_url_list.append(url_link)     # 주소 리스트에 추가

# [중복 제거 - 데이터 정제]
# 대량 수집 시 중복된 글을 피하기 위해 판다스 데이터프레임을 활용합니다.
df_cafe_temp = pd.DataFrame({'title': cafe_title_list, 'url': cafe_url_list})

# 주소(url) 컬럼을 기준으로 중복된 카페 글인지 검사하여 첫 번째 것만 남깁니다.
df_cafe_temp = df_cafe_temp.drop_duplicates(subset=['url'], keep='first')

# 중복 제거된 데이터를 다시 우리가 상세 수집에서 사용할 리스트로 변환
cafe_title_list = df_cafe_temp['title'].tolist()
cafe_url_list = df_cafe_temp['url'].tolist()

print(f"✅ 총 {len(cafe_url_list)}개의 중복 없는 네이버 카페 주소를 확보했습니다.")

✅ 총 540개의 중복 없는 네이버 카페 주소를 확보했습니다.


In [21]:
# --- [네이버 카페 상세 데이터 수집: 본문, 좋아요, 댓글 수, 댓글 내용] ---
# 1. 데이터를 저장할 리스트들 초기화 (네이버 카페용 접두사 cafe_ 사용)
cafe_new_doc = []      # 1. 본문
cafe_like_cnt = []     # 2. 좋아요(공감) 수
cafe_comment_cnt = []  # 3. 댓글 수
cafe_comment_list = [] # 4. 댓글 내용

# 2. cafe_url_list에 저장된 주소 개수만큼 반복문을 실행
# tqdm을 사용하여 카페 데이터 수집 진행률을 시각화합니다.
for i in tqdm(range(len(cafe_url_list)), desc="네이버 카페 상세 수집 중", unit="post"): 
    driver.execute_script(f"window.open('{cafe_url_list[i]}')")
    driver.switch_to.window(driver.window_handles[1]) # 새로 열린 탭으로 제어권 전환
    
    wait = WebDriverWait(driver, 3)
    
    try:
        # [핵심] 네이버 카페는 본문이 'cafe_main'이라는 iframe 안에 들어있습니다.
        # 블로그(mainFrame)와 아이디가 다르므로 주의가 필요합니다.
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, 'cafe_main')))
        time.sleep(0.5) # 카페 메인 프레임 로딩 후 안정적인 파싱을 위해 살짝 대기
        
        html = driver.page_source
        soup = BS(html, 'html.parser')
        
        # --- (1) 본문 텍스트 추출 ---
        try:
            # 카페 에디터 버전에 따라 클래스명이 다를 수 있어 여러 선택자를 지정합니다.
            content = soup.select_one('.se-main-container, .ContentRenderer, .article_viewer').get_text(strip=True)
        except: 
            content = "본문 없음"
        cafe_new_doc.append(content)

        # --- (2) 좋아요(공감) 수 추출 ---
        try:
            # 페이지 하단으로 스크롤하여 좋아요 요소를 활성화
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            
            # 카페의 좋아요 숫자는 em.u_cnt._count 등에 위치합니다.
            like_el = WebDriverWait(driver, 2).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "em.u_cnt._count"))
            )
            like_val = like_el.get_attribute('textContent').strip()
            
            # 숫자가 비어있거나 숫자가 아니면 '0'으로 처리
            if not like_val or not like_val.replace(',', '').isdigit(): 
                like_val = "0"
        except: 
            like_val = "0"
        cafe_like_cnt.append(like_val)

        # --- (3) 댓글 수 추출 ---
        try:
            # 댓글창 영역 근처의 숫자 요소를 찾습니다.
            comment_num_el = driver.find_element(By.CSS_SELECTOR, "strong.num, .comment_count")
            comment_num = comment_num_el.get_attribute('textContent').strip()
            
            # 숫자 외의 문자(쉼표 등) 제거
            comment_num = re.sub(r'[^0-9]', '', comment_num)
            if not comment_num: 
                comment_num = "0"
        except: 
            comment_num = "0"
        cafe_comment_cnt.append(comment_num)

        # --- (4) 댓글 실제 내용 추출 ---
        try:
            # 댓글 버튼(혹은 댓글 영역)을 찾아 클릭하여 내용을 펼칩니다.
            comment_btn = driver.find_element(By.CSS_SELECTOR, "a.button_comment, .btn_comment")
            driver.execute_script("arguments[0].click();", comment_btn) 
            time.sleep(1) # 비동기 로딩 대기
            
            # 카페 댓글 내용 요소(comment_text_view)를 모두 수집
            comments = driver.find_elements(By.CLASS_NAME, "comment_text_view")
            cafe_comment_final = "\n".join([c.text for c in comments]) if comments else "댓글 없음"
        except:
            cafe_comment_final = "댓글 막힘" 
        cafe_comment_list.append(cafe_comment_final)

    except Exception as e:
        # 진행 바 깨짐 방지를 위해 tqdm.write 사용
        tqdm.write(f"⚠️ {i+1}번 카페 글 수집 중 오류 발생")
        cafe_new_doc.append("에러 발생")
        cafe_like_cnt.append("0")
        cafe_comment_cnt.append("0")
        cafe_comment_list.append("에러 발생")

    # 작업 완료 후 탭을 닫고 다시 목록 탭(0번)으로 복귀
    driver.close()
    driver.switch_to.window(driver.window_handles[0])

# 모든 작업 완료 후 드라이버 종료
driver.quit()
print("✅ 수집완료!")

네이버 카페 상세 수집 중: 100%|██████████| 540/540 [22:20<00:00,  2.48s/post]


✅ 수집완료!


In [22]:
# 1. 수집된 카페 리스트들을 하나의 데이터프레임(DataFrame)으로 결합
# 블로그 데이터프레임과 컬럼 순서 및 이름을 동일하게 맞추어 나중에 merge(concat)하기 편하게 만듭니다.
df_cafe = pd.DataFrame({
    'title': cafe_title_list,         # 글 제목
    'url': cafe_url_list,             # 글 주소
    'doc': cafe_new_doc,              # 본문 내용
    'like': cafe_like_cnt,            # 좋아요(공감) 수
    'comment_cnt': cafe_comment_cnt,   # 댓글 수
    'comment_list': cafe_comment_list, # 댓글 내용
    'ch': 'naver',                    # 채널 구분 1 (네이버)
    'ch2': 'cafe'                     # 채널 구분 2 (카페)
})

# 2. 데이터 확인 (제대로 수집되었는지 상위 5개 미리보기)
print("📊 [검토] 네이버 카페 데이터 생성 완료 (상위 5행):")
print(df_cafe.head())

# 3. 파일 저장 (파일명 규칙: 26.02.01~26.04.10_naver_cafe_data.csv)
# 블로그 파일과 동일한 기간을 명시하여 세트로 관리합니다.
file_name_cafe = '26.02.01~26.04.10_naver_cafe_data.csv'

# encoding='utf-8-sig'를 사용하여 한글 깨짐을 방지합니다.
df_cafe.to_csv(file_name_cafe, index=False, encoding='utf-8-sig')

print("-" * 30)
print(f"✅ 파일 저장 완료: {file_name_cafe}")
print(f"총 {len(df_cafe)}개의 카페 게시글 데이터가 저장되었습니다.")

📊 [검토] 네이버 카페 데이터 생성 완료 (상위 5행):
                                   title  \
0        트럼프 미국 이란 전쟁 초토화 선언과호르무즈 해협의 위기   
1     대한민국 '최악의 시나리오'가 왔다. (호르무즈 해협의 위기)   
2  호르무즈 해협개방 됐다는 말 사실일까? 휴전 이후 통항 조건 총정리   
3              [시사 상식] 세계 경제의 목줄,호르무즈 해협   
4          호르무즈 해협내 '대체항로'를 전격 발표 내용정리..   

                                                 url  \
0  https://cafe.naver.com/ssadanicar/180205?art=Z...   
1  https://cafe.naver.com/jaegebal/6226887?art=ZX...   
2  https://cafe.naver.com/cantsb/3251642?art=ZXh0...   
3  https://cafe.naver.com/readbook/2695518?art=ZX...   
4  https://cafe.naver.com/geobuk2/2103829?art=ZXh...   

                                                 doc like comment_cnt  \
0  트럼프 미국 대통령 이란 관련 소식이 전해지면서 전 세계가 긴장하고 있는데 개전 3...    0           4   
1  게시판 안내를 확인해 주세요!*실거래가 게시글에 관한 의견을 묻습니다.https:/...   58          11   
2  게시판 안내를 확인해 주세요!🎁참여자 전원 100% 선물 증정! 👉참여 소요 시간 ...    4          13   
3  북극성 여러분, 안녕하세요😊오늘 오후 7시, 내일부터 적용될 3차 석유 최고 가격이...    4           0   


### 호르무즈 해협을 둘러싼 이슈가 많은 기간(26.02.01~26.04.10) '호르무즈 해협' 데이터 전처리 & 통합

In [23]:
# 특수문자 제거 및 공백 정규화 함수 정의
def clean_text(text):
    # 결측치(NaN) 또는 데이터가 비어있는 경우 처리
    if pd.isna(text) or str(text).strip() == "": 
        return ""
    
    # 한글, 영문, 숫자, 공백만 남기고 모든 특수문자 제거
    cleaned = re.sub("[^0-9a-zA-Zㄱ-ㅎㅏ-ㅣ가-힣 ]", '', str(text))
    
    # 여러 개의 공백을 하나로 줄이고 양끝 공백 제거
    cleaned = ' '.join(cleaned.split())
    return cleaned

try:
    # 1. 데이터 불러오기
    df_blog = pd.read_csv('26.02.01~26.04.10_naver_blog_data.csv')
    df_cafe = pd.read_csv('26.02.01~26.04.10_naver_cafe_data.csv')
    
    # 2. 데이터 통합 (Merge)
    # 두 데이터를 세로로 이어 붙여 하나의 데이터프레임으로 만듭니다.
    df_total = pd.concat([df_blog, df_cafe], ignore_index=True)
    initial_count = len(df_total)
    print(f"✅ 1. 데이터 통합 완료 (총 {initial_count}건)")

    # 3. [정제] 제목, 본문, 댓글 컬럼에 특수문자 제거 적용
    cols_to_clean = ['title', 'doc', 'comment_list']
    
    for col in cols_to_clean:
        if col in df_total.columns:
            # 미리 정의한 clean_text 함수 일괄 적용
            df_total[col] = df_total[col].apply(clean_text)
            
            # 특수문자 제거 후 내용이 사라진 경우 '내용없음'으로 채움 (추후 필터링 위함)
            df_total[col] = df_total[col].replace('', '내용없음')
    
    print("✅ 2. 특수문자 제거 및 내용 정제 완료")

    # 4. [필터링] 유의미한 분석을 위한 노이즈 제거
    # 본문(doc)의 길이가 2글자 미만인 데이터 삭제
    df_total = df_total[df_total['doc'].str.len() >= 2]
    
    # '본문 없음', '내용없음' 등으로 기록된 행 추가 필터링
    df_total = df_total[~df_total['doc'].isin(['본문 없음', '내용없음'])]
    print("✅ 3. 노이즈 및 결측 데이터 필터링 완료")

    # 5. [중복 제거] 본문이 완전히 같은 글은 하나만 남기고 삭제
    # (카페 스크랩 글이나 중복 수집된 게시글 정리)
    df_total = df_total.drop_duplicates(subset=['doc'], keep='first')
    print("✅ 4. 본문 기준 중복 데이터 제거 완료")

    # 6. 인덱스 재설정 및 결과 출력
    # 행이 삭제되어 비어있는 인덱스 번호를 0부터 다시 순서대로 매깁니다.
    df_total = df_total.reset_index(drop=True)
    final_count = len(df_total)

    print(f"📊 [중간 점검 결과]")
    print(f"초기 데이터: {initial_count}건")
    print(f"정제 후 데이터: {final_count}건")
    print(f"제거된 노이즈/중복: {initial_count - final_count}건")
    print("'df_total' 변수에 정제된 데이터가 담겼습니다.")

except Exception as e:
    print(f"❌ 오류 발생: {e}")

✅ 1. 데이터 통합 완료 (총 1080건)
✅ 2. 특수문자 제거 및 내용 정제 완료
✅ 3. 노이즈 및 결측 데이터 필터링 완료
✅ 4. 본문 기준 중복 데이터 제거 완료
📊 [중간 점검 결과]
초기 데이터: 1080건
정제 후 데이터: 1021건
제거된 노이즈/중복: 59건
'df_total' 변수에 정제된 데이터가 담겼습니다.


### 호르무즈 해협을 둘러싼 이슈가 많은 기간(26.02.01~26.04.10) '호르무즈 해협' 데이터 결측값 정리

In [24]:
# 1. 원본 데이터 개수 확인 (앞서 통합한 df_total을 복사해서 사용)
df = df_total.copy()
original_cnt = len(df)

# 2. 결측값 및 단문 정리 (본문 & 댓글 통합 필터링)
# .str.strip()으로 공백을 제거한 후, 순수 글자 수가 2자 이상인 행만 남깁니다.

# 본문(doc) 필터링: 2자 미만 제거
df = df[df['doc'].str.strip().str.len() >= 2]

# 댓글(comment_list) 필터링: 2자 미만 제거 (인사말이나 짧은 이모티콘 등 노이즈 제거)
df = df[df['comment_list'].str.strip().str.len() >= 2]

# 3. '내용없음'으로 표시된 데이터 최종 삭제
# 이전 정제 단계에서 빈 값 대신 채워넣었던 '내용없음' 문자열을 솎아냅니다.
df = df[df['doc'] != '내용없음']
df = df[df['comment_list'] != '내용없음']

# 4. 인덱스 재설정
# 행이 삭제되면 인덱스가 (0, 2, 5...) 처럼 비게 되는데, 
# 이를 0부터 다시 순서대로 매겨야 이후 형태소 분석 루프가 끊기지 않습니다.
df = df.reset_index(drop=True)

# 5. 결과 보고
final_cnt = len(df)
print("-" * 30)
print(f"🧹 결측치 및 단문 정리 완료!")
print(f"📊 정리 전: {original_cnt}개 -> 정리 후: {final_cnt}개")
print(f"🗑️ 제거된 데이터: {original_cnt - final_cnt}개")
print("-" * 30)

# 6. 데이터 최종 확인 (상위 5행 출력)
# 주피터 노트북 환경이라면 display(df.head())를, 일반 환경이라면 print(df.head())를 사용하세요.
try:
    display(df.head())
except NameError:
    print(df.head())

------------------------------
🧹 결측치 및 단문 정리 완료!
📊 정리 전: 1021개 -> 정리 후: 1012개
🗑️ 제거된 데이터: 9개
------------------------------


,title,url,doc,like,comment_cnt,comment_list,ch,ch2
0,미국 이란 전쟁 30일 경과 호르무즈 해협 지도로 본 현 상황,https://blog.naver.com/3f-hoon/224233386857,미국과 이란의 전쟁이 지난 2월 28일에 발발해 한 달이 지났습니다빠르게 끝날 것 ...,16,4,좋은 하루 되세요 늦었지만 답방 왔어요정성글 잘 보고 갑니다,naver,blog
1,호르무즈 해협 통행료가 얼마길래 미국 이란 공동 징수 추진,https://blog.naver.com/totoro3123/224246051445,안녕하세요네이버 경제인플루언서 주식너부리입니다4월 8일현지시각 도널드 트럼프 미국 ...,41,4,호르무즈 해협 통행료가 얼마길래 미국 이란 공동 징수 추진 관련 글 잘 봤어요 좋은...,naver,blog
2,뉴욕 증시 트럼프 발언 속 혼조세 호르무즈 해협 통행료 이슈,https://blog.naver.com/archimall/224239139022,최근 중동 지역의 긴장감이 고조되면서 투자자들의 심리가 얼음판 위를 걷는 듯한 모습...,89,4,정보가 알차서 한 번에 해결됐네요디스커버님 중동 긴장 속 시장 변동성이 컸네요 다행...,naver,blog
3,트럼프 호르무즈 해협 이름 바꾼다 이거 진짜 가능한가,https://blog.naver.com/kimsong8106/224233276505,최근 외신 보도를 보면 도널드 트럼프 대통령이 중동의 핵심 요충지인 호르무즈 해협 ...,25,2,이름 바꾸는 건 쉬운 일이 아니야 역사와 지리의 깊이를 생각해보면 단순한 변화보다는...,naver,blog
4,호르무즈 해협 봉쇄 이유와 영향 유가 상승증시 전망,https://blog.naver.com/dldnsgud91/224201238593,최근 뉴스에서 호르무즈 해협 봉쇄 가능성이라는 표현이 자주 등장하고 있습니다 이 표...,90,3,좋은 정보 감사합니다매우 유익한 정보 입니다손 도장 꾹 찍고 갑니다제 블로그도 방문...,naver,blog


### (~기간2) '호르무즈 해협' 네이버 블로그 크롤링

### (~기간2) '호르무즈 해협' 네이버 카페 크롤링

### (~기간2) '호르무즈 해협' 데이터 통합 & 전처리